# Figure 4: convergence of the training gradient to the flow field

```
python -m extremal_mil.fig4_experiment
```


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import matplotlib.pyplot as plt

from extremal_mil import plotting, theory
from extremal_mil import fig4_experiment as f3
from extremal_mil import convergence as cv

plt.rcParams.update(plotting.RCPARAMS)

FIG_DIR = Path.cwd().parent / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

%config InlineBackend.figure_format = 'svg'

In [ ]:
# What's on disk -- check before assuming a grid below will load.
available = f3.list_available()
print(f"{len(available)} cells in {f3.DEFAULT_DATA_DIR}")
for lam0, ell, d in available:
    print(f"  lam0={lam0:<4g} ell={ell:<6d} d={d:<4d} X={f3.x_gb(ell, d):6.2f} GB")
print(f"\nlambda* = {f3.LAM_STAR:.4f};  u* = phi_alpha(lam0)/rho_true "
      f"= {f3.u_star(f3.LAM0_HIGH_T):.4f} at lam0={f3.LAM0_HIGH_T}, "
      f"{f3.u_star(f3.LAM0_LOW_T):.4f} at lam0={f3.LAM0_LOW_T}")

In [ ]:
COMPONENTS = [("lam", r"$\dot\lambda$"), ("u", r"$\dot u$"), ("rho", r"$\dot\rho$")]



def curve_figure(cells, label, title, path):
    data = [f3.load_curves(*c) for c in cells]
    lam0 = float(data[0]["lam0"])
    assert all(float(x["lam0"]) == lam0 for x in data), "mixed lam0 in one figure"

    colors, _ = plotting.run_colors(np.arange(len(data)))
    fig, axes = plotting.new_row(len(COMPONENTS))
    for ax, (key, ylab) in zip(axes[0], COMPONENTS):
        plotting.zero_line(ax)
        plotting.rn_threshold(ax, float(data[0]["u_star"]))
        for x, c in zip(data, colors):
            ax.plot(x["u"], x[f"emp_{key}"], color=c, lw=1.4, alpha=0.9)
        ax.plot(data[0]["u"], data[0][f"th_{key}"], zorder=3, **plotting.THEORY_STYLE)
        ax.set_ylabel(ylab)

    plotting.finish_row(
        fig, axes, title, "$u$",
        handles=plotting.series_handles(data, colors, label)
        + [plotting.theory_handle(), plotting.rn_handle(r"$u^\star$")])
    fig.savefig(path, bbox_inches="tight")
    plt.show()
    return data

## 1. Main text: sweep $d$ at fixed large $\ell$

$(\lambda_0,\rho_0)=(0.2,1.7)$ -- the trajectory figure's initial
conditions, so the two main-text figures describe one system.
$\lambda_0<\lambda^\star=\sqrt{2\alpha}$, so the softmax has not collapsed
and $\dot\lambda>0$ throughout. $\ell$ is held large, so the panels isolate
convergence in $d$ rather than bag-sampling noise.

In [ ]:
curve_figure(
    [(f3.LAM0_HIGH_T, f3.ELL_MAIN, d) for d in f3.D_VALUES],
    lambda x: rf"$d={int(x['d'])}$ ($n={int(x['n'])}$)",
    None,  # parameters are stated in the caption
    FIG_DIR / "fig4.pdf")